In [0]:
data = [
    (101,"Rahul",75000),
    (102,"Priya",85000),
    (103,"Amit",65000)
]

df = spark.createDataFrame(
    data,
    ["emp_id","name","salary"]
)

In [0]:
df.write.format("delta") \
.save("/tmp/employees_delta")

In [0]:
delta_df = spark.read.format("delta") \
.load("/tmp/employees_delta")

display(delta_df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:

df.write.format("delta") \
.saveAsTable("employees")

In [0]:
%sql
select * from employees



emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql
CREATE TABLE IF NOT EXISTS employees_delta
(
    emp_id INT,
    name STRING,
    salary INT
)
USING DELTA
;
INSERT INTO employees_delta
VALUES
(101,'Rahul',75000),
(102,'Priya',85000)

num_affected_rows,num_inserted_rows
2,2


In [0]:
updates = [

(102,"Priya",90000),
(104,"Sneha",70000)

]

updates_df = spark.createDataFrame(
    updates,
    ["emp_id","name","salary"]
)


In [0]:

from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/employees_delta"
)

delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql DESCRIBE HISTORY employees
 

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T10:25:01.000Z,146284169126344,azuser7211_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3379337785578204),45aaa050-3f17-45d2-836c-370209397779,0617-102318-xci7n1o-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1307)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",1) \
.load("/tmp/employees_delta")
 
display(df)

emp_id,name,salary
101,Rahul,75000
103,Amit,65000
102,Priya,90000
104,Sneha,70000
